In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn as skt
import xgboost as xgb
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
import dtale


previous_application_df = pd.read_parquet(cfg.CLEANS_DIR / "previous_application.train-cleaned.parquet")



In [ ]:
previous_application_df.head(5)

In [ ]:
#creating columns before aggregation
previous_application_df["diff_application_credit"] = previous_application_df["amt_application"] - previous_application_df["amt_credit"]
previous_application_df["ratio_credit_to_goods"] = previous_application_df["amt_credit"] / (previous_application_df["amt_goods_price"].replace(0,np.nan))
previous_application_df["total_interest_charged"] = (previous_application_df["amt_annuity"] * previous_application_df["cnt_payment"]) - previous_application_df["amt_credit"]
previous_application_df["ratio_credit_to_annuity"]= previous_application_df["amt_credit"] / (previous_application_df["amt_annuity"].replace(0,np.nan))


In [ ]:
instalament_df = pd.read_parquet(cfg.PROCESSED_DIR / "installments_payments.train-processed.parquet")
previous_application_df= previous_application_df.merge(instalament_df,how="left",on= "id_prev")
previous_application_df.head()

In [ ]:

previous_application_to_pivot_df= previous_application_df.drop(columns="id_prev")
mask_no_final= previous_application_df["flag_last_application_for_the_contract"] == "N"
previous_application_to_pivot_df= previous_application_to_pivot_df.loc[~mask_no_final]
previous_application_to_pivot_df= previous_application_to_pivot_df.drop(columns=["flag_last_application_for_the_contract"])

previous_application_to_pivot_df.sort_values(["id_curr","days_decision"],inplace=True,ascending=False)
last_three= previous_application_to_pivot_df.groupby("id_curr").head(3)

In [ ]:
last_three = last_three.copy()
last_three["loan_order"] = last_three.groupby("id_curr").cumcount() + 1
df_wide = last_three.pivot(index="id_curr", columns="loan_order")
df_wide.columns =[f"{col}_prev_{rank}" for col, rank in df_wide.columns]
df_wide= df_wide.reset_index()
dtale.show(df_wide)

In [ ]:
previous_application_df["name_contract_status"]= previous_application_df["name_contract_status"].str.lower()
previous_application_df["code_reject_reason"]= previous_application_df["code_reject_reason"].str.lower()
previous_application_df= pd.get_dummies(previous_application_df, columns=["name_contract_status","code_reject_reason"])
dtale.show(previous_application_df.head(5))

In [ ]:



#we will calculate aggregations per client for differents tables so we will separate dictionaries per table
agg_from_prev_app_dict= {

    #saving the ammount of contract
    "id_prev" : ["count"],
    
    #for log transformated we want to catch the mean and the std (avoiding the impact of the heavy tail from this columns)
    "log_amt_credit": ["mean","std"],   
    "log_amt_application": ["mean","std"],
    "log_amt_down_payment": ["mean","std"],
    "log_amt_goods_price": ["mean","std"],
    "log_amt_annuity": ["mean","std"],
    "log_total_interest_charged" : ["mean","std"],

    #for non transformated columns we want to catch the representative values and the acumulated
    "amt_credit": ["max", "min","median","sum"],
    "amt_application": ["max", "min","median","sum"],
    "amt_down_payment": ["max", "min","median","sum"],
    "amt_goods_price": ["max", "min","median","sum"], 
    "amt_annuity": ["max", "min","median"],
    "total_interest_charged": ["max", "min","median"],

    #others_monetary
    "diff_application_credit": ["max","mean","min","median","sum"],
    "log_diff_application_credit": ["max","mean","min"],
    "rate_down_payment": ["max","mean","std","min","median"],
    "ratio_credit_to_goods" : ["max","mean","std","min","median"],
    "ratio_credit_to_annuity" : ["max","mean","std","min","median"],

    #categoricals
    "code_reject_reason_limit" : ["sum"],
    "name_contract_status_approved": ["mean","sum"],
    "name_contract_status_canceled": ["mean","sum"],
    "name_contract_status_refused": ["mean","sum"],
    "amt_annuity_and_cnt_payment_are_missing" :["mean","sum"],
    "amt_down_payment_is_missing" : ["mean","sum"],
    "nflag_insured_on_approval" : ["mean","sum"],
    "days_and_insurance_information_are_missing": ["mean","sum"],
    "amt_goods_price_is_missing" : ["mean","sum"],
    "rate_down_payment_is_missing" : ["mean","sum"],


    #counters
    "days_decision":["mean","min","max"],
    "cnt_payment":["mean","min","max","sum"]
}

In [ ]:
agg_from_instalment_payment_dict= {
    "instalments_potentially_on_going" : ["sum"],
    "instalments_is_potentially_incomplete_sequence" : ["mean","sum"],
    "instalments_dead_tail_length" : ["mean","max"],
    "instalments_amt_instalment_sum" : ["mean","sum","max"],
    "instalments_amt_payment_sum" : ["mean", "sum", "max"],
    "instalments_days_of_delinquency_max": ["max"],
    "instalments_days_of_delinquency_mean": ["mean", "max"],
    "instalments_days_in_advance_max":["max"],
    "instalments_days_in_advance_mean": ["mean", "max"],
    "instalments_is_delinquency_sum" : ["sum"] ,
    "instalments_is_delinquency_mean" : ["mean"],
    "instalments_repeated_for_underpayment_sum" : ["sum"],
    "instalments_repeated_for_underpayment_mean" : ["mean"],
    "instalments_repeated_for_reschedule_sum" : ["sum"],
    "instalments_repeated_for_reschedule_mean" : ["mean"],
    "instalments_diff_expected_received_sum" : ["sum"]
}


In [ ]:
final_dict_for_agg= agg_from_prev_app_dict | agg_from_instalment_payment_dict

In [ ]:

previous_application_df["log_amt_credit"] = np.log1p(previous_application_df["amt_credit"])
previous_application_df["log_amt_application"] = np.log1p(previous_application_df["amt_application"])
previous_application_df["log_amt_annuity"] = np.log1p(previous_application_df["amt_annuity"])
previous_application_df["log_amt_down_payment"] = np.log1p(previous_application_df["amt_down_payment"])
previous_application_df["log_amt_goods_price"] = np.log1p(previous_application_df["amt_goods_price"])
previous_application_df["log_diff_application_credit"] = previous_application_df["log_amt_application"] - previous_application_df["log_amt_credit"]
previous_application_df["log_total_interest_charged"] = np.log1p(previous_application_df["total_interest_charged"])



agg_metrics_df= previous_application_df.groupby("id_curr").agg(final_dict_for_agg)




In [ ]:
agg_metrics_df.columns= [f"{col[0]}_{col[1]}" for col in agg_metrics_df.columns]
agg_metrics_df= agg_metrics_df.reset_index()

In [ ]:
agg_metrics_df.rename(columns={"id_prev_count": "applications_count"})
agg_metrics_df.head()


In [ ]:
agg_metrics_df.rename(columns={"id_prev_count": "applications_count"},inplace=True)
previous_application_ready_to_merge= df_wide.merge(agg_metrics_df,on="id_curr",how="left")

In [ ]:
cols_to_fix = [
    "instalments_potentially_on_going_sum",
    "instalments_is_potentially_incomplete_sequence_sum",
    "instalments_is_potentially_incomplete_sequence_mean",
    "instalments_is_delincuency_sum",
    "instalments_is_delincuency_mean"
]

for col in cols_to_fix:
    if col in previous_application_ready_to_merge.columns:
        previous_application_ready_to_merge[col] = pd.to_numeric(previous_application_ready_to_merge[col], errors='coerce')

In [ ]:

previous_application_ready_to_merge.to_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments.parquet", index=False)

In [ ]:
instalament_df = pd.read_parquet(cfg.PROCESSED_DIR / "installments_payments.train-processed-2.parquet")
instalament_df.head()

In [44]:
previous_application_df = pd.read_parquet(cfg.CLEANS_DIR / "previous_application.train-cleaned.parquet")
#creating columns before aggregation
previous_application_df["diff_application_credit"] = previous_application_df["amt_application"] - previous_application_df["amt_credit"]
previous_application_df["ratio_credit_to_goods"] = previous_application_df["amt_credit"] / (previous_application_df["amt_goods_price"].replace(0,np.nan))
previous_application_df["total_interest_charged"] = (previous_application_df["amt_annuity"] * previous_application_df["cnt_payment"]) - previous_application_df["amt_credit"]
previous_application_df["implied_interest_rate"] = (previous_application_df["amt_annuity"] * previous_application_df["cnt_payment"]) / previous_application_df["amt_credit"].replace(0, np.nan)
previous_application_df["ratio_credit_to_annuity"]= previous_application_df["amt_credit"] / (previous_application_df["amt_annuity"].replace(0,np.nan))

instalament_df = pd.read_parquet(cfg.PROCESSED_DIR / "installments_payments.train-processed-2.parquet")



#results = instalament_df[["instalments_id_curr","instalments_mean_ratio_last_year", "instalments_mean_ratio_last_two_years","instalments_mean_historical_payment_rate","instalments_payment_tendence"]].drop_duplicates()

#instalament_df=instalament_df.drop(columns=["instalments_mean_ratio_last_year","instalments_mean_ratio_last_two_years","instalments_mean_historical_payment_rate","instalments_payment_tendence","instalments_id_curr"])



previous_application_df= previous_application_df.merge(instalament_df,how="left",on= "id_prev")
previous_application_to_pivot_df= previous_application_df.drop(columns="id_prev")
mask_no_final= previous_application_df["flag_last_application_for_the_contract"] == "N"
previous_application_to_pivot_df= previous_application_to_pivot_df.loc[~mask_no_final]
previous_application_to_pivot_df= previous_application_to_pivot_df.drop(columns=["flag_last_application_for_the_contract"])

previous_application_to_pivot_df.sort_values(["id_curr","days_decision"],inplace=True,ascending=False)
last_three= previous_application_to_pivot_df.groupby("id_curr").head(1)
last_three = last_three.copy()
last_three["loan_order"] = last_three.groupby("id_curr").cumcount() + 1
df_wide = last_three.pivot(index="id_curr", columns="loan_order")
df_wide.columns =[f"{col}_prev_{rank}" for col, rank in df_wide.columns]
df_wide= df_wide.reset_index()
#previous_application_df["name_contract_status"]= previous_application_df["name_contract_status"].str.lower()
previous_application_df["code_reject_reason"]= previous_application_df["code_reject_reason"].str.lower()
#previous_application_df["name_contract_type"] = previous_application_df["name_contract_type"].str.lower()
previous_application_df = pd.get_dummies(previous_application_df, columns=[ "code_reject_reason"])

#we will calculate aggregations per client for differents tables so we will separate dictionaries per table
agg_from_prev_app_dict= {

    #saving the ammount of contract
    "id_prev" : ["count"],
    
    #for log transformated we want to catch the mean and the std (avoiding the impact of the heavy tail from this columns)
    "log_amt_credit": ["mean","std"],   
    "log_amt_application": ["mean","std"],
    "log_amt_down_payment": ["mean","std"],
    "log_amt_goods_price": ["mean","std"],
    "log_amt_annuity": ["mean","std"],
    "log_total_interest_charged" : ["mean","std"],
    "instalments_completion_ratio" : ["mean","std"],
 
    

    #for non transformated columns we want to catch the representative values and the acumulated
    "amt_credit": ["max", "min","median","sum"],
    "amt_application": ["max", "min","median","sum"],
    "amt_down_payment": ["max", "min","median","sum"],
    "amt_goods_price": ["max", "min","median","sum"],
    "amt_annuity": ["max", "min","median"],
    "total_interest_charged": ["max", "min","median"],
    "implied_interest_rate" : ["max", "min","mean","std"],
    

    #others_monetary
    "diff_application_credit": ["max","mean","min","sum","median"], #
    "log_diff_application_credit": ["max","mean","min"],
    "rate_down_payment": ["max","mean","min","median","std"], #
    "ratio_credit_to_goods" : ["max","mean","median","min","std"], # 
    "ratio_credit_to_annuity" : ["max","mean","min","median","std"], #
    #categoricals
    #"name_contract_status_approved": ["mean","sum"],
    #"name_contract_status_canceled": ["mean","sum"],
    #"name_contract_status_refused": ["mean","sum"],
    #"name_contract_type_cash loans": ["mean", "sum"], 
    #"name_contract_type_consumer loans": ["mean", "sum"],
    #"name_contract_type_revolving loans": ["mean", "sum"], 
    "amt_annuity_and_cnt_payment_are_missing" :["mean","sum"],
    "amt_down_payment_is_missing" : ["mean","sum"],
    "nflag_insured_on_approval" : ["mean","sum"],
    "days_and_insurance_information_are_missing": ["mean","sum"],
    "amt_goods_price_is_missing" : ["mean","sum"],
    "rate_down_payment_is_missing" : ["mean","sum"],


    #counters
    "days_decision":["mean","min","max"],
    "cnt_payment":["mean","max","sum"]
}



agg_from_instalment_payment_dict= {
    #"instalments_days_instalment_max":["min","max"],
    #"instalments_days_instalment_min":["min"],
    #"instalments_days_instalment_mean":["mean"],
    "instalments_amount_of_versions_in_sequence" : ["mean","max","sum"],
    "instalments_potentially_on_going" : ["sum"],
    "instalments_dead_tail_length" : ["mean","max"],
    "instalments_amt_instalment_sum" : ["mean","sum","max"],
    "instalments_amt_payment_sum" : ["mean", "sum", "max"],
    "instalments_days_of_delinquency_max": ["max"],
    "instalments_days_of_delinquency_mean": ["mean", "max"],
    "instalments_extra_instalament_sum":["sum"],
    "instalments_extra_instalament_mean":["mean"],
    "instalments_days_in_advance_max":["max"],
    "instalments_days_of_underpayment_max":["max"],
    "instalments_days_in_advance_mean": ["mean", "max"],
    "instalments_is_delinquency_sum" : ["sum"] ,
    "instalments_is_delinquency_mean" : ["mean"],
    "instalments_diff_expected_received_sum" : ["sum"]
}

final_dict_for_agg= agg_from_prev_app_dict | agg_from_instalment_payment_dict


previous_application_df["log_amt_credit"] = np.log1p(previous_application_df["amt_credit"])
previous_application_df["log_amt_application"] = np.log1p(previous_application_df["amt_application"])
previous_application_df["log_amt_annuity"] = np.log1p(previous_application_df["amt_annuity"])
previous_application_df["log_amt_down_payment"] = np.log1p(previous_application_df["amt_down_payment"])
previous_application_df["log_amt_goods_price"] = np.log1p(previous_application_df["amt_goods_price"])
previous_application_df["log_diff_application_credit"] = previous_application_df["log_amt_application"] - previous_application_df["log_amt_credit"]
previous_application_df["log_total_interest_charged"] = np.log1p(previous_application_df["total_interest_charged"])



agg_metrics_df= previous_application_df.groupby("id_curr").agg(final_dict_for_agg)

agg_metrics_df.columns= [f"{col[0]}_{col[1]}" for col in agg_metrics_df.columns]

agg_metrics_df.rename(columns={"id_prev_count": "applications_count"},inplace=True)
agg_metrics_df= agg_metrics_df.reset_index()

agg_metrics_df["global_approval_ratio"] = agg_metrics_df["amt_credit_sum"] / agg_metrics_df["amt_application_sum"].replace(0, np.nan)
agg_metrics_df["days_decision_spread"] = agg_metrics_df["days_decision_max"] - agg_metrics_df["days_decision_min"]

previous_application_ready_to_merge= df_wide.merge(agg_metrics_df,on="id_curr",how="left")

cols_to_fix = [
    "instalments_potentially_on_going_sum",
    "instalments_is_potentially_incomplete_sequence_sum",
    "instalments_is_potentially_incomplete_sequence_mean",
    "instalments_is_delinquency_sum",
    "instalments_is_delinquency_mean"
]

time_window_df= pd.read_parquet(cfg.PROCESSED_DIR / "time_window_instalments.parquet")


previous_application_ready_to_merge= previous_application_ready_to_merge.merge(time_window_df,how="left",on="id_curr")

for col in cols_to_fix:
    if col in previous_application_ready_to_merge.columns:
       previous_application_ready_to_merge[col] = pd.to_numeric(previous_application_ready_to_merge[col], errors='coerce')


previous_application_ready_to_merge.to_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments_time_window.parquet", index=False)

c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [ ]:
previous_application_ready_to_merge = pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments-2.parquet")
dtale.show(previous_application_ready_to_merge.head(5))


KeyError: 'id_prev'

In [ ]:
previous_application_df = pd.read_parquet(cfg.CLEANS_DIR / "previous_application.train-cleaned.parquet")
instalament_df = pd.read_parquet(cfg.PROCESSED_DIR / "installments_payments.train-processed-2.parquet")
what= previous_application_df.merge(instalament_df,how="left",on= "id_prev")
dtale.show(what.head(50))